Based on the SLURM logs after running the batch_labeling job in the hopes of fixing the missing "orange" data, we have a "Good News / Bad News" situation.
- The good news: my previous assumption about the node counts was slightly off, **the node counts are actually different** (74,137 for Manhattan vs. 31,036 for Pittsburgh).

    Meaning: `pickle.load()` is indeed bringing in a new graph object.

- The bad news: **the "Pittsburgh Bug" is likely a spatial mismatch, not a loading failure.**

---

## What the Logs Reveal:

### 1. The Component Count Suspicion
* **Manhattan:** 74,137 nodes $\rightarrow$ 1 component.
* **Pittsburgh:** 31,036 nodes $\rightarrow$ 1 component.
While it's possible for a city graph to be perfectly connected, seeing exactly **1 component** in a city-scale graph (especially Pittsburgh, which is bisected by three rivers) after switching to **Undirected** logic is a bit "too perfect." 

Possible reasons:
- Our `get_connectivity_map` is too aggressive.
- The graph is somehow truncated.
- The `SymbolicSolver` might believe every node is reachable from every other node, yet it still fails to find the specific landmarks.

### 2. The Coordinate "Dead Zone"
The high count of **Contradictory (400)** in Pittsburgh suggests that the solver is successfully extracting landmarks but failing to find a valid path between them.

### 3. Verification of the "Pittsburgh Stats"
Looking at these specific samples from our latest Pittsburgh log:
* `Sample 106 -> Answerable | Cands: 1`
* `Sample 52 -> Contradictory | Cands: 0`

When `Cands` (Candidates) is **0**, it means the solver's reachability logic returned an empty set for the final instruction step. Given that Pittsburgh's file size is less than half of Manhattan's, but the label distribution is heavily skewed toward failure, we need to check the **Coordinate Reference System (CRS)**.

---

This problem is called: a **"State Persistence Bug".**

I fixed it by forcing the __init__ in OracleEngine to receive prefix and city_name instead of getting them from config (OracleEngine(G, poi_path, prefix, city_name)), and that is to make it impossible for the Oracle to be "lazy." It can no longer look at a global config and guess where it is. It has to build its spatial index using exactly what I(batch_labeling.py) handed it in that specific moment.

Problem That Remains: a **"Yield Drop"**

### 📉 The Yield Comparison

| City | Original Yield (Answerable) | New Yield (Answerable) | Change |
| :--- | :--- | :--- | :--- |
| **Pittsburgh** | 778 (76.1%) | 321 (31.3%) | **-45.8%** |
| **Philadelphia** | 1,035 (80.9%) | 454 (35.5%) | **-45.4%** |

### 🔍 Why did the percentages shift so drastically?

My goal was to fix a data-balancing issue (LLM mask types), but in doing so, the "Forensic Auditing" (Section 3 of the original version's report) has become too aggressive.

#### 1. The "Ambiguous" Explosion
In Philly, we now have **357 Ambiguous** cases. In the original report, Ambiguous was lower. 
* **Reason:** The new speed-optimized `resolve_landmark` logic might be finding *too many* candidates. If a landmark is resolved to multiple nodes and we don't have a tie-breaking "Salience" check like we did before, the solver will mark it `Ambiguous` instead of `Answerable`.

#### 2. The "Contradictory" Wall (The 400s)
Both cities are hovering around **400-460 Contradictory**. 
* **Reason:** Since we verified the spatial bounds (Pittsburgh POIs are in Pittsburgh), the problem is **Connectivity**.
* In the original report, we mentioned a **1500m Range Contradiction**. If the new speed-optimized code uses a smaller default buffer or a stricter `scc_lookup` (the Undirected components), it is likely rejecting landmarks that the original version "accepted" by being more lenient with the search radius.

#### 3. "Prefix" and "Node ID" mismatches
In the log, we see: `Sample None -> Label: Answerable`.
* **The "None" issue:** The new batch script is failing to pull the `rvs_sample_number`. This suggests the internal dataframe structure of the JSONL might be slightly different than what the script expects (`row.get('rvs_sample_number')`).

---

### 🛠️ How to get Our 70%+ Yield back?

To maintain the speed optimizations but restore the "Gold" quality, we're planning to check these three specific logic points in the `SymbolicSolver` and `OracleEngine`:

**A. Restore the "Success Radius"**
Originally, we used **250m for Philly** and **100m for Pittsburgh**. 
* **Check:** Is `self.search_radius` in `SymbolicSolver` actually receiving these values from `config`? If it's defaulting to a small value (e.g., 50m) to save speed, we will lose 40% of the data to "Contradictory" because the agent "can't reach" the landmark.

**B. Landmark Resolution (The "Salience" missing link)**
the report mentions tuning the **Salience Ratio (0.5 to 0.7)**. 
* **Check:** Does your new `OracleEngine` use this ratio? If the new logic doesn't filter by salience, a landmark like "Post Office" will return 50 candidates (Ambiguous). If you apply the salience filter, it might prune that down to 1 primary candidate (Answerable).

**C. The Connectivity Check**
You switched to Undirected logic. While this is "King" for reachability, if your graph has many "micro-islands" (nodes with no edges), snapping to them causes an instant `Contradictory`.
* **Fix:** In your `fast_snap` function, ensure you are snapping only to nodes that belong to the **Largest Connected Component**.

### Summary of the "Unpleasant" Numbers
The numbers you are seeing aren't a "bug" in the code execution (the script is running perfectly now), they are a **policy change** in the solver. Your current solver is "pessimistic"—it would rather say Contradictory/Ambiguous than risk an incorrect path. Your original solver was "optimistic."

**To fix the Philly LLM "masking" issue without losing yield:** You should keep the isolated Slurm setup but re-introduce the **Salience Filtering** and **City-Specific Radii** from your report into the `OracleEngine` initialization.